# Módulo 07 · Notebook 2: El pipeline "de producción" de House Prices

Ahora juntamos **todo** lo del curso en un solo objeto:

```
 datos crudos ─> ColumnTransformer ───────────────────────────────┐
                  ├─ numéricas:  imputar(mediana) → escalar        │
                  ├─ categóricas: imputar(moda) → one-hot          ├─> modelo (log del precio) ─> USD
                  └─ ordinales:  imputar(moda) → ordinal(Po<…<Ex)  │
                                                                   ┘
```

Al final **guardamos el pipeline** en `models/`, listo para que la API del módulo 08 lo use.

In [ ]:
import json
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

from intro_analitica import config
from intro_analitica.datos import cargar_house_prices_limpio
from intro_analitica.metricas import metricas_regresion

sns.set_theme(style="whitegrid")
df = cargar_house_prices_limpio()

X = df[config.FEATURES_MODELO]
y = df[config.TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=config.SEMILLA)
print(f"Train: {X_train.shape} · Test: {X_test.shape}")

> Ahora solo dividimos en **train y test**: la validación la hace la **validación cruzada** dentro del train.

---
## 1. `ColumnTransformer`: un tratamiento por tipo de columna

In [ ]:
NUMERICAS = config.FEATURES_NUMERICAS
ORDINALES = {"KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"]}
CATEGORICAS = [c for c in config.FEATURES_CATEGORICAS if c not in ORDINALES]

preprocesamiento = ColumnTransformer(
    [
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]), NUMERICAS),
        (
            "cat",
            Pipeline(
                [
                    ("imputar", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            CATEGORICAS,
        ),
        (
            "ord",
            Pipeline(
                [
                    ("imputar", SimpleImputer(strategy="most_frequent")),
                    ("ordinal", OrdinalEncoder(categories=list(ORDINALES.values()), handle_unknown="use_encoded_value", unknown_value=-1)),
                ]
            ),
            list(ORDINALES),
        ),
    ],
    verbose_feature_names_out=False,
)
preprocesamiento

**¿Por qué imputar si los datos ya están limpios?** Porque en **producción** pueden llegar datos con huecos. El pipeline debe ser robusto a lo que **pueda** llegar, no solo a lo que vimos en el entrenamiento.

Veamos qué produce:

In [ ]:
X_transformado = preprocesamiento.fit_transform(X_train)
print("Columnas de entrada:", X_train.shape[1], "→ columnas de salida:", X_transformado.shape[1])
pd.DataFrame(X_transformado, columns=preprocesamiento.get_feature_names_out()).head(3).round(2)

## 2. El target en logaritmo, de forma automática

En el módulo 06 hacíamos `np.log1p(y)` para entrenar y `np.expm1(pred)` para volver a dólares. `TransformedTargetRegressor` lo hace solo:

In [ ]:
def con_log(modelo):
    return TransformedTargetRegressor(regressor=modelo, func=np.log1p, inverse_func=np.expm1)


pipeline = Pipeline([("preprocesamiento", preprocesamiento), ("modelo", con_log(LinearRegression()))])
pipeline

---
## 3. Comparar modelos con validación cruzada (sin *leakage*)

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=config.SEMILLA)
scoring = {"MAE": "neg_mean_absolute_error", "MAPE": "neg_mean_absolute_percentage_error", "R2": "r2"}

candidatos = {
    "Lineal + log": con_log(LinearRegression()),
    "Ridge + log": con_log(Ridge(alpha=1.0)),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=config.SEMILLA, n_jobs=-1),
    "Gradient Boosting + log": con_log(HistGradientBoostingRegressor(random_state=config.SEMILLA)),
}

filas = []
for nombre, modelo in candidatos.items():
    pipeline.set_params(modelo=modelo)
    res = cross_validate(pipeline, X_train, y_train, cv=kfold, scoring=scoring, return_train_score=True)
    filas.append(
        {
            "modelo": nombre,
            "MAE_cv": -res["test_MAE"].mean(),
            "MAPE_cv_%": -res["test_MAPE"].mean() * 100,
            "MAPE_std": res["test_MAPE"].std() * 100,
            "R2_cv": res["test_R2"].mean(),
            "MAE_train": -res["train_MAE"].mean(),
        }
    )
comparacion = pd.DataFrame(filas).set_index("modelo").sort_values("MAPE_cv_%")
comparacion.round(2)

---
## 4. Afinar el mejor con `GridSearchCV`

Recuerda la notación `paso__subpaso__parametro`. Como el modelo está **envuelto** en `TransformedTargetRegressor`, la ruta es `modelo__regressor__<parámetro>`.

In [ ]:
pipeline.set_params(modelo=con_log(HistGradientBoostingRegressor(random_state=config.SEMILLA)))

grilla = {
    "modelo__regressor__learning_rate": [0.03, 0.1],
    "modelo__regressor__max_leaf_nodes": [15, 31],
    "modelo__regressor__min_samples_leaf": [10, 20],
    "modelo__regressor__l2_regularization": [0.0, 1.0],
}
busqueda = GridSearchCV(pipeline, grilla, cv=kfold, scoring="neg_mean_absolute_percentage_error", n_jobs=-1)
busqueda.fit(X_train, y_train)

print("Mejores hiperparámetros:")
for k, v in busqueda.best_params_.items():
    print(f"  {k.split('__')[-1]:<18} = {v}")
print(f"MAPE CV: {-busqueda.best_score_ * 100:.2f} %")

---
## 5. Evaluación final en test

In [ ]:
mejor = busqueda.best_estimator_
pred_test = mejor.predict(X_test)
metricas_test = metricas_regresion(y_test, pred_test)
metricas_test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_test, pred_test, alpha=0.4)
lim = [y_test.min(), y_test.max()]
axes[0].plot(lim, lim, "r--", label="predicción perfecta")
axes[0].set(title="Real vs. predicho (test)", xlabel="Real (USD)", ylabel="Predicho (USD)")
axes[0].legend()
error_pct = (pred_test / y_test - 1) * 100
sns.histplot(error_pct, bins=40, ax=axes[1])
axes[1].axvline(-15, color="red", ls="--")
axes[1].axvline(15, color="red", ls="--")
axes[1].set(title=f"Error % ({(np.abs(error_pct) <= 15).mean():.0%} de las casas dentro de ±15 %)", xlabel="Error %")
plt.tight_layout()
plt.show()

---
## 6. Guardar el pipeline (el "artefacto" del modelo)

Guardamos el pipeline **completo** (preprocesamiento + modelo) con `joblib`, junto con un archivo de **metadatos**: cuándo se entrenó, con qué variables y qué métricas obtuvo.
Sin metadatos, en tres meses nadie sabrá qué es `modelo_final_v2.joblib`.

In [ ]:
config.asegurar_carpetas()
ruta_modelo = config.MODELS_DIR / "house_price_model.joblib"
ruta_metadata = config.MODELS_DIR / "house_price_model.json"

joblib.dump(mejor, ruta_modelo)
metadata = {
    "modelo": "HistGradientBoostingRegressor + log(target)",
    "version": "1.0.0",
    "entrenado_en": datetime.now().isoformat(timespec="seconds"),
    "features_numericas": NUMERICAS,
    "features_categoricas": CATEGORICAS + list(ORDINALES),
    "hiperparametros": {k.split("__")[-1]: v for k, v in busqueda.best_params_.items()},
    "metricas_test": metricas_test,
    "n_train": len(X_train),
}
ruta_metadata.write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
print("", ruta_modelo.relative_to(config.RAIZ), f"({ruta_modelo.stat().st_size / 1024:.0f} KB)")
print("", ruta_metadata.relative_to(config.RAIZ))

## 7. Cargar y predecir una casa nueva

Este es **exactamente** el flujo que hará la API: cargar el archivo y llamar a `predict` con datos **crudos**.

In [ ]:
modelo_cargado = joblib.load(ruta_modelo)

casa_nueva = pd.DataFrame(
    [
        {
            "LotArea": 9000, "OverallQual": 7, "OverallCond": 5, "YearBuilt": 2005, "YearRemodAdd": 2006,
            "GrLivArea": 1800, "TotalBsmtSF": 1000, "FullBath": 2, "BedroomAbvGr": 3, "GarageCars": 2,
            "Neighborhood": "CollgCr", "BldgType": "1Fam", "HouseStyle": "2Story", "KitchenQual": "Gd",
        }
    ]
)
print(f"Precio estimado: USD {modelo_cargado.predict(casa_nueva)[0]:,.0f}")

In [ ]:
# ¿Y si llega un barrio que el modelo nunca vio? handle_unknown="ignore" evita el error
casa_rara = casa_nueva.assign(Neighborhood="BarrioNuevo")
print(f"Barrio desconocido → USD {modelo_cargado.predict(casa_rara)[0]:,.0f} (no falla [OK])")

> Todo este notebook está resumido en `07-Pipelines/scripts/train.py`, que usa `src/intro_analitica/pipeline.py`. **El notebook es para entender; el script es para repetir.**

---
## Ejercicios guiados (20 min)

1. **Nueva variable en el pipeline:** agrega `TotalSF` a las numéricas (¡también en `X`!) y repite la comparación de la sección 3. ¿Mejora?
2. **Random Forest con log:** agrega `"Random Forest + log": con_log(RandomForestRegressor(...))` a los candidatos. ¿El log le ayuda a los árboles tanto como a la lineal?
3. **Robustez:** predice con `casa_nueva` pero con `GrLivArea = None`. ¿Qué pasa? ¿Qué paso del pipeline lo resuelve?
4. **Importancia por permutación:** usa `sklearn.inspection.permutation_importance(mejor, X_test, y_test, n_repeats=5)` para ver qué variables originales importan más.

Soluciones en `ejercicios/soluciones/solucion_07_house_prices.py`.

In [ ]:
# Tu código aquí